# Modelling a gallium arsenide surface

This example shows how to use the atomistic simulation environment or ASE for short,
to set up and run a particular calculation of a gallium arsenide surface.
ASE is a Python package to simplify the process of setting up,
running and analysing results from atomistic simulations across different simulation codes.
For more details on the integration DFTK provides with ASE,
see Atomistic simulation environment.

In this example we will consider modelling the (1, 1, 0) GaAs surface separated by vacuum.

Parameters of the calculation. Since this surface is far from easy to converge,
we made the problem simpler by choosing a smaller `Ecut` and smaller values
for `n_GaAs` and `n_vacuum`.
More interesting settings are `Ecut = 15` and `n_GaAs = n_vacuum = 20`.

In [1]:
miller = (1, 1, 0)   # Surface Miller indices
n_GaAs = 2           # Number of GaAs layers
n_vacuum = 4         # Number of vacuum layers
Ecut = 5             # Hartree
kgrid = (4, 4, 1);   # Monkhorst-Pack mesh

Use ASE to build the structure:

In [2]:
using ASEconvert

a = 5.6537  # GaAs lattice parameter in Ångström (because ASE uses Å as length unit)
gaas = ase.build.bulk("GaAs", "zincblende"; a)
surface = ase.build.surface(gaas, miller, n_GaAs, 0, periodic=true);

    CondaPkg Found dependencies: /home/runner/.julia/packages/ASEconvert/I2wvg/CondaPkg.toml
    CondaPkg Found dependencies: /home/runner/.julia/packages/PythonCall/Nr75f/CondaPkg.toml
    CondaPkg Dependencies already up to date


Get the amount of vacuum in Ångström we need to add

In [3]:
d_vacuum = maximum(maximum, surface.cell) / n_GaAs * n_vacuum
surface = ase.build.surface(gaas, miller, n_GaAs, d_vacuum, periodic=true);

Write an image of the surface and embed it as a nice illustration:

In [4]:
ase.io.write("surface.png", surface * pytuple((3, 3, 1)), rotation="-90x, 30y, -75z")

Python: None

<img src="https://docs.dftk.org/stable/surface.png" width=500 height=500 />

Use the `pyconvert` function from `PythonCall` to convert to an AtomsBase-compatible system.
These two functions not only support importing ASE atoms into DFTK,
but a few more third-party data structures as well.
Typically the imported `atoms` use a bare Coulomb potential,
such that appropriate pseudopotentials need to be attached in a post-step:

In [5]:
using DFTK
system = attach_psp(pyconvert(AbstractSystem, surface);
                    Ga="hgh/pbe/ga-q3.hgh",
                    As="hgh/pbe/as-q5.hgh")

FlexibleSystem(As₂Ga₂, periodic = TTT):
    bounding_box      : [ 3.99777        0        0;
                                0  3.99777        0;
                                0        0  21.2014]u"Å"

    Atom(Ga, [       0,        0,  8.48055]u"Å")
    Atom(As, [ 3.99777,  1.99888,  9.89397]u"Å")
    Atom(Ga, [ 1.99888,  1.99888,  11.3074]u"Å")
    Atom(As, [ 1.99888, 6.96512e-16,  12.7208]u"Å")

      .---------.  
     /|         |  
    * |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |  As     |  
    | |   Ga    |  
    | |         |  
    | |        As  
    | |         |  
    | |         |  
    Ga|         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | |         |  
    | .---------.  
    |/         /   
    *---------*    


We model this surface with (quite large a) temperature of 0.01 Hartree
to ease convergence. Try lowering the SCF convergence tolerance (`tol`)
or the `temperature` or try `mixing=KerkerMixing()`
to see the full challenge of this system.

In [6]:
model = model_DFT(system;
                  functionals=PBE(),
                  temperature=0.001, smearing=DFTK.Smearing.Gaussian())
basis = PlaneWaveBasis(model; Ecut, kgrid)

scfres = self_consistent_field(basis; tol=1e-4, mixing=LdosMixing());

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -16.58944639308                   -0.58    5.3    1.27s
  2   -16.72540113843       -0.87       -1.01    1.0    352ms
  3   -16.73063970627       -2.28       -1.57    2.3    363ms
  4   -16.73126104183       -3.21       -2.16    1.1    533ms
  5   -16.73132808609       -4.17       -2.60    1.7    316ms
  6   -16.73133576651       -5.11       -2.92    2.0    347ms
  7   -16.73114869619   +   -3.73       -2.67    2.0    369ms
  8   -16.73113188720   +   -4.77       -2.62    2.3    363ms
  9   -16.73123061672       -4.01       -2.77    1.9    331ms
 10   -16.73132522729       -4.02       -3.18    1.0    252ms
 11   -16.73133931452       -4.85       -3.69    1.6    292ms
 12   -16.73133980297       -6.31       -3.95    1.6    291ms
 13   -16.73133959847   +   -6.69       -3.88    1.9    326ms
 14   -16.73134018500       -6.23       -4.54    1.0    250ms


In [7]:
scfres.energies

Energy breakdown (in Ha):
    Kinetic             5.8593940 
    AtomicLocal         -105.6100355
    AtomicNonlocal      2.3494753 
    Ewald               35.5044300
    PspCorrection       0.2016043 
    Hartree             49.5614554
    Xc                  -4.5976603
    Entropy             -0.0000035

    total               -16.731340185000